In [3]:
import numpy as np
import scipy.constants as sc
from pywarpx import picmi, warpx
from dataclasses import dataclass, field

In [2]:
C    = sc.c
E_C  = sc.e
M_P  = sc.m_p
M_E  = sc.m_e
EV   = sc.eV
MU0  = 4 * np.pi * 1e-7
EPS0 = sc.epsilon_0

In [14]:
@dataclass
class SingleCoil3DConfig:

    # Stream params
    n_stream:   float = 1.0e17     # m^-3
    v_drift:    float = 5.0e5      # m/s, magnitude (flow toward -x)
    T_e_eV:     float = 10.0       # electron fluid temperature
    T_i_eV:     float = 10.0       # ion thermal spread on injection

    # Coil params (coil sits at x=0, axis along x)
    I:          float = 1e5        # A
    dia:        float = 0.5        # m

    # Grid
    L:          float = 2.5        # m (half-extent, all axes)
    N:          int   = 64         # cells per axis

    # Numerical stability (mirrors coil_2d.py)
    eta_bg:     float = 1.0e-7     # background resistivity (Ohm·m)
    eta_coil:   float = 1.0e3      # coil island resistivity (Ohm·m)
    eta_H:      float = 3.0e-3     # hyper-resistivity (Ohm·m^3)
    n_floor_frac: float = 0.05     # n_floor as fraction of n_stream
    substeps:   int   = 100        # B-field substeps

    # Derived — not set by user
    R_coil:     float = field(init=False)   # coil radius (m)
    dx:         float = field(init=False)   # cell width (m)
    eps:        float = field(init=False)   # wire regularization width (~dx)
    m_i:        float = field(init=False)
    rho:        float = field(init=False)   # upstream mass density
    P_ram:      float = field(init=False)   # ram pressure
    B_ref:      float = field(init=False)   # |B| at r_CF
    r_CF:       float = field(init=False)   # Chapman-Ferraro standoff (m)
    Omega_ci:   float = field(init=False)   # ion cyclotron freq at r_CF
    d_i:        float = field(init=False)   # ion inertial length
    dt:         float = field(init=False)   # outer timestep
    n_floor:    float = field(init=False)   # absolute density floor

    def __post_init__(self):
        MU0 = sc.mu_0

        self.R_coil = self.dia / 2
        self.dx     = 2 * self.L / self.N
        self.eps    = self.dx                        # one cell, matches coil_2d convention

        self.m_i    = sc.m_p
        self.rho    = self.n_stream * self.m_i
        self.P_ram  = self.rho * self.v_drift**2

        # On-axis B for a circular loop: B_x(r) = mu0*I*R^2 / (2*(R^2+r^2)^(3/2))
        # Solve P_ram = B^2 / (2*mu0) numerically for r_CF
        self.r_CF = np.sqrt((MU0 * self.I * self.R_coil**2 / (2 * np.sqrt(2 * MU0 * self.P_ram)))**(2/3) - self.R_coil**2)
        if self.r_CF < 0 or not np.isreal(self.r_CF):
            raise ValueError(f"r_CF is imaginary — P_ram too large for coil to stand off the flow. Increase I or reduce n_stream/v_drift.")

        self.B_ref    = MU0 * self.I * self.R_coil**2 / (2 * (self.R_coil**2 + self.r_CF**2)**1.5)
        self.Omega_ci = sc.e * self.B_ref / self.m_i
        self.d_i      = sc.c / np.sqrt(self.n_stream * sc.e**2 / (sc.epsilon_0 * self.m_i))
        self.n_floor  = self.n_floor_frac * self.n_stream

        # Timestep: min of 1/50 cyclotron period and 0.4*dx/v_e_th
        v_e_th    = np.sqrt(self.T_e_eV * sc.eV / sc.m_e)
        dt_cyclo  = 1.0 / (50.0 * self.Omega_ci)
        dt_efluid = 0.4 * self.dx / v_e_th
        self.dt   = min(dt_cyclo, dt_efluid)

        print(f"P_ram       = {self.P_ram:.3e} Pa")
        print(f"r_CF        = {self.r_CF*100:.2f} cm  (predicted standoff)")
        print(f"B at r_CF   = {self.B_ref*1e4:.2f} G")
        print(f"d_i (n_inf) = {self.d_i*100:.2f} cm  (ion inertial length upstream)")
        print(f"Omega_ci    = {self.Omega_ci:.3e} rad/s  -> 1/Omega_ci = {1/self.Omega_ci:.3e} s")

In [ ]:
cfg = SingleCoil3DConfig()

P_ram       = 4.182e+01 Pa
r_CF        = 68.19 cm  (predicted standoff)
B at r_CF   = 102.52 G
d_i (n_inf) = 72.01 cm  (ion inertial length upstream)
Omega_ci    = 9.820e+05 rad/s  -> 1/Omega_ci = 1.018e-06 s
0.010251533303954037


In [ ]:
grid = picmi.Cartesian3DGrid(
    number_of_cells=[cfg.N, cfg.N, cfg.N],
    lower_bound=[-cfg.L]*3,
    upper_bound=[cfg.L]*3,
    lower_boundary_conditions=["neumann", "neumann"],
    upper_boundary_conditions=["neumann", "neumann"],
    lower_boundary_conditions_particles=["absorbing", "absorbing"],
    upper_boundary_conditions_particles=["absorbing", "absorbing"],
    warpx_max_grid_size=16,
)

In [ ]:
import os, sys, pathlib

parent = pathlib.Path(os.pardir)
parent_parent = parent.parent
sys.path.append(parent_parent)

In [17]:
from bext import analytic
analytic.POLYWELL_COILS = [('x', 1, 1)]
A_external = analytic.build_aext_expressions(I=cfg.I, dia=cfg.dia, offset=0.0)
print(A_external)

ModuleNotFoundError: No module named 'bext'

In [ ]:


A_external = 

solver = picmi.HybridPICSolver(
    grid=grid,
    Te=cfg.T_e_eV,
    n0=cfg.n_stream,
    gamma=5.0/3.0,
    n_floor=0.05 * cfg.n_stream,                 # 5% of upstream — caps 1/n amplification in the deepening cavity
    plasma_resistivity=cfg.eta_bg,               # uniform; coil islands emulated via callback
    plasma_hyper_resistivity=3.0e-3,         # Ohm·m^3; overdamps grid-Nyquist whistlers at peak |B|~0.13T
    holmstrom_vacuum_region=True,            # suppress Hall/pressure terms in the cavity
    substeps=100,                           # dt_sub ≈ 3e-11 s; clears whistler CFL at peak |B| with ~3x margin
    A_external=A_external,
    do_external_diva_cleaning=False,         # A is analytically div-free
)

MAX_STEPS = 1000
sim = picmi.Simulation(
    solver=solver,
    max_steps=cfg.ma,
    verbose=True,
    particle_shape="linear",
    warpx_grid_type="collocated",   # recommended for hybrid
)